# Whisper LoRA 파인튜닝 - 외국인 한국어 발음 인식

## 목표
- Whisper-small 모델을 LoRA로 파인튜닝
- 외국인 발음을 "들리는 대로" 변환하도록 학습
- 적은 데이터(30개)로 효율적인 학습

## 1. 환경 설정 및 라이브러리 설치

In [1]:
import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [2]:
# 필요한 라이브러리 설치 (최초 1회)
# !pip install --upgrade transformers datasets accelerate peft bitsandbytes
# !pip install --upgrade librosa soundfile jiwer evaluate

In [3]:
import os
import torch
import pandas as pd
import numpy as np
from pathlib import Path

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch version: 2.8.0+cu128
CUDA available: True
GPU: NVIDIA L40S
GPU Memory: 47.8 GB


## 2. 경로 설정 및 데이터 로드

In [4]:
# 경로 설정 (필요에 따라 수정)
WORK_DIR = "data/들리는json"
AUDIO_DIR = "data/들리는wav"  # 음성 파일 폴더 경로 (수정 필요)
EXCEL_PATH = "data/들리는 대로 변환 정리 문서.xlsx"

os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(f"{WORK_DIR}/checkpoints", exist_ok=True)

print(f"작업 디렉토리: {WORK_DIR}")

작업 디렉토리: data/들리는json


In [5]:
# !pip install openpyxl

In [6]:
# 엑셀 데이터 로드
df = pd.read_excel(EXCEL_PATH)
print(f"총 샘플 수: {len(df)}")
print(f"\n컬럼: {df.columns.tolist()}")
print(f"\n처음 5개 데이터:")
df.head()

총 샘플 수: 31

컬럼: ['음성 번호', '원본 문장', '들리는 문장']

처음 5개 데이터:


,음성 번호,원본 문장,들리는 문장
0,07961-F-1-VI-B-ATQ021-1768242,제 방이 작은 편이에요. 방 안에 침대하고 책상이 있어요. 그리고 오창도 하나 있고...,제 방이 작근 펴니에요. 방 아네 침대하고 채쌍이 이써요. 그리고 오창도 하나 이꼬...
1,07458-F-1-VI-B-PDT001-1879662,한국어를 잘하려면 무작정 책으로만 공부해서는 안 돼. 제일 중요한 것은 한국인하고 ...,한꾸거를 자라려면 무작청 채그로만 곤부해서는 안 돼. 제일 충요한 거슨 한구긴하꼬 ...
2,01666-F-2-VI-B-SSO001-0386569,"사람들은 행복하게 살기 위해 가족, 친구, 건강, 사랑, 일, 돈 등을 말합니다. ...","사람트른 행보카게 살기 위해 가족, 친구, 건강, 사랑, 일, 돈 등을 마랍니다. ..."
3,01689-F-1-VI-B-ATQ028-0785142,집은 구할 때 가장 중요하게 생각하는 거는 햇빛이 많은 것입니다. 왜냐하면 햇빛이 ...,지븐 구할 때 가장 중요하게 생가카는 거는 해삐치 마는 거십니다. 왜냐하면 해삐치 ...
4,01535-F-3-VI-B-SPT006-0432276,저는 어렸을 때 학교에 지각한 적이 있어요. 그때는 교동이 스트레스 때문에 그리고 ...,저는 어려쓸 때 하꾜에 지가칸 저기 이써요. 그때는 교통이 츨스 때무네 그리고 저는...


In [7]:
# 음성 파일 존재 확인
def find_audio_file(audio_id, audio_dir):
    """음성 번호로 파일 찾기"""
    possible_extensions = ['.wav', '.mp3', '.m4a', '.flac']
    
    for ext in possible_extensions:
        # 정확한 파일명
        path = os.path.join(audio_dir, f"{audio_id}{ext}")
        if os.path.exists(path):
            return path
        
        # 부분 매칭
        for file in os.listdir(audio_dir) if os.path.exists(audio_dir) else []:
            if audio_id in file:
                return os.path.join(audio_dir, file)
    
    return None

# 파일 매칭 확인 (AUDIO_DIR이 존재하면)
if os.path.exists(AUDIO_DIR):
    matched = 0
    for idx, row in df.iterrows():
        audio_path = find_audio_file(row['음성 번호'], AUDIO_DIR)
        if audio_path:
            matched += 1
    print(f"매칭된 음성 파일: {matched}/{len(df)}")
else:
    print(f"⚠️ 음성 폴더가 없습니다: {AUDIO_DIR}")
    print("음성 파일을 업로드한 후 AUDIO_DIR 경로를 수정해주세요.")

매칭된 음성 파일: 30/31


## 3. Whisper 모델 및 프로세서 로드

In [8]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration

MODEL_NAME = "openai/whisper-small"

print(f"모델 로딩: {MODEL_NAME}")
processor = WhisperProcessor.from_pretrained(MODEL_NAME)
model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)

# 한국어 설정 (새로운 방식)
model.generation_config.language = "korean"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None  # 자동 설정되도록

print(f"모델 파라미터 수: {model.num_parameters():,}")
print("모델 로드 완료!")

모델 로딩: openai/whisper-small


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

모델 파라미터 수: 241,734,912
모델 로드 완료!


## 4. LoRA 설정

In [9]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# LoRA 설정
lora_config = LoraConfig(
    r=16,  # LoRA rank (작은 데이터셋에는 8-16 권장)
    lora_alpha=32,  # LoRA alpha
    target_modules=["q_proj", "v_proj"],  # 적용할 모듈
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_2_SEQ_LM"
)

# LoRA 적용
model = get_peft_model(model, lora_config)

# 학습 가능한 파라미터 확인
model.print_trainable_parameters()

trainable params: 1,769,472 || all params: 243,504,384 || trainable%: 0.7267


## 5. 데이터셋 준비

In [10]:
import librosa
import soundfile as sf
from torch.utils.data import Dataset, DataLoader

class KoreanPronunciationDataset(Dataset):
    def __init__(self, dataframe, audio_dir, processor, target_sr=16000):
        self.df = dataframe
        self.audio_dir = audio_dir
        self.processor = processor
        self.target_sr = target_sr
        
        # 유효한 샘플만 필터링
        self.valid_samples = []
        for idx, row in self.df.iterrows():
            audio_path = find_audio_file(row['음성 번호'], audio_dir)
            if audio_path:
                self.valid_samples.append({
                    'audio_path': audio_path,
                    'transcript': row['들리는 문장']  # 들리는 대로 학습!
                })
        
        print(f"유효한 샘플 수: {len(self.valid_samples)}")
    
    def __len__(self):
        return len(self.valid_samples)
    
    def __getitem__(self, idx):
        sample = self.valid_samples[idx]
        
        # 오디오 로드
        audio, sr = librosa.load(sample['audio_path'], sr=self.target_sr)
        
        # 프로세서로 입력 특성 추출
        input_features = self.processor(
            audio, 
            sampling_rate=self.target_sr, 
            return_tensors="pt"
        ).input_features.squeeze(0)
        
        # 라벨 토큰화
        labels = self.processor.tokenizer(
            sample['transcript'],
            return_tensors="pt"
        ).input_ids.squeeze(0)
        
        return {
            "input_features": input_features,
            "labels": labels
        }

## 6. Data Collator 정의

In [18]:
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # 입력 특성 패딩
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        
        # 라벨 패딩
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        
        # 패딩 토큰을 -100으로 변경 (loss 계산에서 무시)
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        
        # BOS 토큰 제거 (있다면)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
        
        batch["labels"] = labels
        
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

## 7. 평가 메트릭 정의

In [19]:
import evaluate
from jiwer import wer, cer

# G2P for PER
try:
    from g2pk import G2p
    g2p = G2p()
    USE_G2P = True
except:
    print("G2P를 사용할 수 없습니다. PER은 계산되지 않습니다.")
    USE_G2P = False

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    
    # -100을 패드 토큰으로 변환
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    
    # 디코딩
    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    
    # WER, CER 계산
    wer_score = wer(label_str, pred_str)
    cer_score = cer(label_str, pred_str)
    
    results = {
        "wer": wer_score * 100,
        "cer": cer_score * 100,
    }
    
    # PER 계산 (G2P 사용 가능시)
    if USE_G2P:
        pred_phonemes = [g2p(text) for text in pred_str]
        label_phonemes = [g2p(text) for text in label_str]
        per_score = cer(label_phonemes, pred_phonemes)
        results["per"] = per_score * 100
    
    return results

print("평가 메트릭 함수 정의 완료")

평가 메트릭 함수 정의 완료


## 8. 학습 설정

In [28]:
# 데이터셋 생성
dataset = KoreanPronunciationDataset(df, AUDIO_DIR, processor)

# 학습/검증 분리 (90/10)
train_size = int(0.9 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = torch.utils.data.random_split(
    dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

print(f"학습 데이터: {len(train_dataset)}")
print(f"검증 데이터: {len(val_dataset)}")

유효한 샘플 수: 30
학습 데이터: 27
검증 데이터: 3


In [29]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

training_args = Seq2SeqTrainingArguments(
    output_dir=f"{WORK_DIR}/checkpoints",
    
    # 배치 및 에폭 설정 (작은 데이터셋용)
    per_device_train_batch_size=10,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    num_train_epochs=20,  # 작은 데이터셋이므로 에폭 증가
    
    # 학습률 설정
    learning_rate=1e-4,  # LoRA에는 약간 높은 학습률
    warmup_steps=50,
    
    # 평가 및 저장
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="cer",
    greater_is_better=False,
    
    # 생성 설정
    predict_with_generate=True,
    generation_max_length=225,
    
    # 최적화
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=2,
    
    # 로깅
    logging_dir=f"{WORK_DIR}/logs",
    logging_steps=10,
    report_to="none",  # wandb 사용시 "wandb"로 변경
)

print("학습 설정 완료")
print(f"총 에폭: {training_args.num_train_epochs}")
print(f"배치 크기: {training_args.per_device_train_batch_size}")
print(f"학습률: {training_args.learning_rate}")

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


학습 설정 완료
총 에폭: 20
배치 크기: 10
학습률: 0.0001


## 9. Trainer 초기화 및 학습

In [30]:
# Trainer 초기화
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
)

print("Trainer 초기화 완료")

Trainer 초기화 완료


In [ ]:
# 학습 시작
print("="*60)
print("학습 시작!")
print("="*60)

trainer.train()

## 10. 모델 저장

In [34]:
# LoRA 어댑터 저장
SAVE_PATH = f"{WORK_DIR}/no7_model_whisper-small-korean-pronunciation-lora2"
model.save_pretrained(SAVE_PATH)
processor.save_pretrained(SAVE_PATH)

print(f"모델 저장 완료: {SAVE_PATH}")

모델 저장 완료: data/들리는json/no7_model_whisper-small-korean-pronunciation-lora2


## 11. 저장된 모델 다시 로드하기

In [35]:
# 나중에 모델 다시 로드하는 방법
from peft import PeftModel

def load_finetuned_model(lora_path, base_model_name="openai/whisper-small"):
    """저장된 LoRA 모델 로드"""
    # 베이스 모델 로드
    base_model = WhisperForConditionalGeneration.from_pretrained(base_model_name)
    processor = WhisperProcessor.from_pretrained(base_model_name)
    
    # LoRA 어댑터 적용
    model = PeftModel.from_pretrained(base_model, lora_path)
    
    # 한국어 설정
    model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(language="korean", task="transcribe")
    
    return model, processor

# 사용 예시
loaded_model, loaded_processor = load_finetuned_model(SAVE_PATH)
print("모델 로드 함수 정의 완료")

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

모델 로드 함수 정의 완료
